# Nokken-ontwerp — Groep Colin Cools & Lander Cortens

**Specificatie nr. 21** (zie `specificatie.md`).

| Onderdeel | Waarde |
|---|---|
| Heffing 45° → 80° | **+10 mm** |
| Heffing 90° → 130° | **+15 mm** |
| Heffing 130° → 180° | **−25 mm** |
| Equivalente massa $m$ | **21 kg** |
| Dempingsverhouding $\zeta$ | **0,09** |
| Drukkracht 100° → 130° | lineair 0 N → 700 N |
| Trekkracht 130° → 170° | constant 350 N |
| Cyclustijd $T$ | **1 s** (→ 60 rpm) |

Deze notebook bevat:

1. **Bewegingswet** opbouwen uit segmenten (cycloïde / 5e-graads).
2. **Geometrie**: steekcirkel $R_0$ (Kloomok-Muffley), volger-radius $R_r$, excentriciteit $e$, drukhoek $\alpha$, kromtestraal $\rho$, nok-contour.
3. **Externe kracht** opbouwen.
4. **Veer-ontwerp** ($k_v$, $F_{v0}$) zodat $N(\theta) > 0$ overal.
5. **Krachten, vermogen en koppel** + structuur voor **vliegwiel** en **motor**.
6. **Trillingsanalyse** (1-DOF flexibele volger, single-rise + multi-rise, FFT van het versnellingsspectrum).

> **Tip voor het examen**: pas in de eerste cel de spec-parameters aan om snel een gevoeligheidsanalyse te doen (vraag 10 / vraag 2 / vraag 4 / vraag 5).


In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import TransferFunction, lsim, tf2ss

plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["figure.dpi"] = 100


## 1. Specificatie en globale parameters

Pas hier de parameters aan voor gevoeligheids-experimenten.

In [ ]:
# === Spec 21 ===
T_cycle = 1.0                # cyclustijd [s]
rpm     = 60.0 / T_cycle     # rotaties per minuut
omega   = 2*np.pi / T_cycle  # hoeksnelheid van de nok [rad/s]

# Heffingen (start_hoek, eind_hoek, start_lift, eind_lift, motionlaw)
# motionlaw: 1=dwell, 2=3e-graad (min RMS acc), 3=harmonisch, 4=cycloide, 5=5e-graads, 6=7e-graads
heffingen = [
    # 0°  – 45°  : dwell op 0
    (0,   45,   0,   0,   1),
    # 45° – 80°  : +10 mm  (rise)        — cycloïde
    (45,  80,   0,  10,   4),
    # 80° – 90°  : dwell op 10
    (80,  90,  10,  10,   1),
    # 90° – 130° : +15 mm  (rise)        — cycloïde
    (90, 130,  10,  25,   4),
    # 130°– 180° : −25 mm  (fall)        — cycloïde
    (130,180,  25,   0,   4),
    # 180°–360°  : dwell op 0
    (180,360,   0,   0,   1),
]
# Keuze van bewegingswetten — verdediging:
# - Alle drie de actieve segmenten zitten tussen dwells. Een cycloïde heeft
#   d²S/dθ² = 0 aan zowel begin als einde van het segment, waardoor er geen
#   versnellings-sprong is op de rise->dwell overgang. Dat is precies het type
#   sprong dat residual vibration veroorzaakt (zie §7).
# - Harmonisch is hier expliciet ongeschikt: d²S/dθ² springt van 0 (in dwell) naar
#   een eindige waarde aan het begin van de rise.
# - 7e-graads polynoom geeft een nog gladder jerk-profiel, ten koste van ~20%
#   hogere piek-versnelling. We laten in §2.1 de vergelijking zien, en in §7 de
#   trade-off in trillingsamplitude. Cycloïde wint op piek-acceleratie en is dus
#   onze keuze als kost en piek-contactkracht relevant zijn.

# Externe statische krachten (positief = drukkracht volger->nok-richting; negatief = trekkracht)
# (start_hoek, eind_hoek, start_kracht_N, eind_kracht_N)
ext_load_segments = [
    (100, 130,    0,  700),    # lineair toenemende drukkracht 0 -> 700 N
    (130, 170, -350, -350),    # constante trekkracht 350 N (negatief, want trekkracht)
]

# Volger-dynamica (uit spec)
mass_eq       = 21.0       # equivalente massa van de volger [kg]
zeta_follower = 0.09       # dempingsverhouding

# Resolutie
dtheta   = 0.01            # graden
theta_deg = np.arange(0, 360, dtheta)
theta     = theta_deg * np.pi/180

print(f"rpm = {rpm:.1f}  |  omega = {omega:.2f} rad/s  |  steekjes = {len(theta_deg)}")


## 2. Bewegingswet — segmenten samenstellen

**Keuze: volle cycloïde voor alle drie de actieve segmenten.**

Het volledige profiel is van het type *stijgen-dwell-stijgen-dalen-dwell*. Elk actief segment ligt
dus **tussen dwells**. Daarom is het cruciaal dat $\ddot S = 0$ aan beide uiteinden van elk segment
— anders ontstaat er een **versnellingssprong** op de overgang, en die sprong is de directe
oorzaak van *residual vibration* (zie §7).

| Wet | $\ddot S$ aan rand | Piek $\ddot S$ (genormaliseerd) | Continue jerk? |
|---|---|---|---|
| Harmonisch (3) | **≠ 0** — sprong! | $\pi^2/2 \approx 4{,}93$ | nee |
| Cycloïde (4) | **= 0** ✓ | $2\pi \approx 6{,}28$ | nee (sprong in jerk) |
| 5e-graads (5) | **= 0** ✓ | $5{,}77$ | ja |
| 7e-graads (6) | **= 0** ✓ + jerk = 0 | $7{,}51$ | ja, plus continue snap |

Harmonisch valt af wegens de acceleratiesprong. Tussen cycloïde, 5e- en 7e-graads is er een
**trade-off** tussen piek-versnelling (=> piek-contactkracht, piek-koppel) en gladheid van de
jerk (=> hoe sterk de hogere harmonischen het systeem aanslaan).

**Verdediging van de keuze (examenvraag 4)**:

- *Tegenover harmonisch:* cycloïde elimineert de acceleratiesprong op de dwell-overgangen.
- *Tegenover 7e-graads:* cycloïde geeft ~16% lagere piek-versnelling. Bij een **inertiegedreven** ontwerp
  (zoals dit — $m = 21$ kg op 60 rpm is voornamelijk inertie) bepaalt $\max|\ddot S|$ direct de
  piek-contactkracht en het piek-koppel, dus de motorgrootte en de Hertzdruk op het contact.
- *Tegenover 5e-graads:* het verschil in piek-acc tussen cycloïde en 5e-graads is klein, maar
  5e-graads heeft een **discontinue snap** (vierde afgeleide springt niet, maar de FFT laat zien
  dat 5e-graads wel zwakker is in hogere harmonischen — beperkt verschil voor ons systeem).

In **§2.1** vergelijken we de wetten kwantitatief op één segment, in **§7.5** vergelijken we hun
trillingsantwoord.

In [ ]:
lift    = np.zeros_like(theta)   # mm
vel_deg = np.zeros_like(theta)   # mm/deg
acc_deg = np.zeros_like(theta)   # mm/deg^2

def addMotionSegment(startangle, endangle, startlift, endlift, motionlaw):
    assert endangle > startangle, "End angle must be larger than start angle"
    i0 = int(round(startangle/dtheta))
    i1 = int(round(endangle/dtheta))
    beta = endangle - startangle                # graden
    L0, L1 = startlift, endlift
    L = L1 - L0
    x = np.linspace(0, 1, i1 - i0)              # genormaliseerd

    if motionlaw == 1:        # dwell
        assert L == 0, "Dwell vereist startlift==endlift"
        s   = L0 * np.ones_like(x)
        v_d = np.zeros_like(x)
        a_d = np.zeros_like(x)
    elif motionlaw == 2:      # 3rd-order (min RMS acc)
        s   = L0 + L*(3*x**2 - 2*x**3)
        v_d = L/beta * (6*x - 6*x**2)
        a_d = L/beta**2 * (6 - 12*x)
    elif motionlaw == 3:      # harmonisch
        s   = L0 + L*(1 - np.cos(np.pi*x))/2
        v_d = L/beta * np.sin(np.pi*x)*np.pi/2
        a_d = L/beta**2 * np.cos(np.pi*x)*np.pi**2/2
    elif motionlaw == 4:      # volle cycloide
        s   = L0 + L*(x - np.sin(2*np.pi*x)/(2*np.pi))
        v_d = L/beta * (1 - np.cos(2*np.pi*x))
        a_d = 2*np.pi*L/beta**2 * np.sin(2*np.pi*x)
    elif motionlaw == 5:      # 5e-graads
        s   = L0 + L*(6*x**5 - 15*x**4 + 10*x**3)
        v_d = L/beta * (30*x**4 - 60*x**3 + 30*x**2)
        a_d = L/beta**2 * (120*x**3 - 180*x**2 + 60*x)
    elif motionlaw == 6:      # 7e-graads
        s   = L0 + L*(-20*x**7 + 70*x**6 - 84*x**5 + 35*x**4)
        v_d = L/beta * (-140*x**6 + 420*x**5 - 420*x**4 + 140*x**3)
        a_d = L/beta**2 * (-840*x**5 + 2100*x**4 - 1680*x**3 + 420*x**2)
    else:
        raise ValueError(f"Onbekende motionlaw {motionlaw}")

    lift[i0:i1]    = s
    vel_deg[i0:i1] = v_d
    acc_deg[i0:i1] = a_d

# Bouw de bewegingswet
for seg in heffingen:
    addMotionSegment(*seg)

# Omrekenen naar SI-eenheden voor de volger
# vel_deg [mm/deg] -> vel [mm/rad]; acc_deg [mm/deg^2] -> acc [mm/rad^2]
vel = vel_deg * 180/np.pi
acc = acc_deg * (180/np.pi)**2

# vel in mm/s en acc in mm/s^2 (vermenigvuldig met omega en omega^2)
vel_t = vel * omega           # mm/s
acc_t = acc * omega**2        # mm/s^2


In [ ]:
# Plot de bewegingswet
fig, ax = plt.subplots(3, 1, figsize=(9, 7), constrained_layout=True)
ax[0].plot(theta_deg, lift);            ax[0].set_ylabel("S [mm]");          ax[0].grid()
ax[1].plot(theta_deg, vel);             ax[1].set_ylabel("dS/dθ [mm/rad]");  ax[1].grid()
ax[2].plot(theta_deg, acc);             ax[2].set_ylabel("d²S/dθ² [mm/rad²]");ax[2].set_xlabel("θ [deg]"); ax[2].grid()
for a in ax: a.set_xlim(0, 360)
fig.suptitle("Bewegingswet — heffing, snelheid, versnelling (genormaliseerd op θ)")
plt.show()

print(f"max |S|   = {np.max(np.abs(lift)):.2f} mm")
print(f"max |v_t| = {np.max(np.abs(vel_t)):.1f} mm/s")
print(f"max |a_t| = {np.max(np.abs(acc_t)):.1f} mm/s² = {np.max(np.abs(acc_t))/9810:.2f} g")


### 2.1 Vergelijking van bewegingswetten op één segment

We vergelijken kwantitatief op het **langste actieve segment** (90°–130°, +15 mm) hoe harmonisch,
cycloïde, 5e-graads en 7e-graads zich gedragen. We tonen $S, \dot S, \ddot S$ én de **jerk** $\dddot S$.

> **Onthoud voor het examen**: dit is precies de trade-off die hoort bij examenvraag 4.

In [ ]:
# Vergelijk op het langste actieve segment: 90°-130°, +15 mm
seg_start, seg_end, L0, L1 = 90, 130, 10, 25
beta = (seg_end - seg_start) * np.pi/180
x = np.linspace(0, 1, 500)
L = L1 - L0

profiles = {}
# Harmonisch
profiles["Harmonisch"] = dict(
    S  = L0 + L*(1 - np.cos(np.pi*x))/2,
    Sp = L/beta * np.sin(np.pi*x)*np.pi/2,
    Sa = L/beta**2 * np.cos(np.pi*x)*np.pi**2/2,
    Sj = -L/beta**3 * np.sin(np.pi*x)*np.pi**3/2,
)
# Cycloïde
profiles["Cycloïde"] = dict(
    S  = L0 + L*(x - np.sin(2*np.pi*x)/(2*np.pi)),
    Sp = L/beta * (1 - np.cos(2*np.pi*x)),
    Sa = 2*np.pi*L/beta**2 * np.sin(2*np.pi*x),
    Sj = 4*np.pi**2*L/beta**3 * np.cos(2*np.pi*x),
)
# 5e-graads
profiles["5e-graads"] = dict(
    S  = L0 + L*(6*x**5 - 15*x**4 + 10*x**3),
    Sp = L/beta * (30*x**4 - 60*x**3 + 30*x**2),
    Sa = L/beta**2 * (120*x**3 - 180*x**2 + 60*x),
    Sj = L/beta**3 * (360*x**2 - 360*x + 60),
)
# 7e-graads
profiles["7e-graads"] = dict(
    S  = L0 + L*(-20*x**7 + 70*x**6 - 84*x**5 + 35*x**4),
    Sp = L/beta * (-140*x**6 + 420*x**5 - 420*x**4 + 140*x**3),
    Sa = L/beta**2 * (-840*x**5 + 2100*x**4 - 1680*x**3 + 420*x**2),
    Sj = L/beta**3 * (-4200*x**4 + 8400*x**3 - 5040*x**2 + 840*x),
)

fig, ax = plt.subplots(4, 1, figsize=(10, 10), constrained_layout=True)
labels = ["Heffing S [mm]", "dS/dθ [mm/rad]", "d²S/dθ² [mm/rad²]", "d³S/dθ³ [mm/rad³] (jerk)"]
keys   = ["S", "Sp", "Sa", "Sj"]
for name, p in profiles.items():
    for i, k in enumerate(keys):
        ax[i].plot(x, p[k], label=name)
for i, lab in enumerate(labels):
    ax[i].set_ylabel(lab); ax[i].grid(); ax[i].legend(loc="best", fontsize=8)
    ax[i].axhline(0, c="k", lw=0.5)
ax[-1].set_xlabel("genormaliseerde positie x = (θ - θ_start)/β")
fig.suptitle(f"Vergelijking bewegingswetten op segment {seg_start}°-{seg_end}° (+{L1-L0} mm)")
plt.show()

print("Piek-versnelling (mm/rad²) per wet:")
for name, p in profiles.items():
    print(f"   {name:12s}: max|d²S/dθ²| = {np.max(np.abs(p['Sa'])):.1f}  ;  max|jerk| = {np.max(np.abs(p['Sj'])):.1f}")


## 3. Cam-geometrie

### 3.1 Minimale steekcirkel $R_0$ — Kloomok-Muffley

We bepalen voor elke **stijgende** of **dalende** segment de relatie tussen $R_0$ en de maximale
drukhoek $\alpha_{\max}$. Vuistregel: $|\alpha| < 30°$ voor een translatie-volger.

In [ ]:
def Kloomok_Muffley_R0(beta_deg, L0, L1, motionlaw, R0_vec=None):
    # Geeft alpha_max(R0) voor een segment.
    if R0_vec is None:
        R0_vec = np.arange(0.5, 150, 0.5)
    L = L1 - L0
    beta = beta_deg * np.pi/180
    x = np.linspace(0, 1, 200)

    if motionlaw == 1:
        return R0_vec, np.zeros_like(R0_vec)
    if motionlaw == 2:
        s, v = L0 + L*(3*x**2 - 2*x**3), L/beta*(6*x - 6*x**2)
    elif motionlaw == 3:
        s, v = L0 + L*(1 - np.cos(np.pi*x))/2, L/beta*np.sin(np.pi*x)*np.pi/2
    elif motionlaw == 4:
        s, v = L0 + L*(x - np.sin(2*np.pi*x)/(2*np.pi)), L/beta*(1 - np.cos(2*np.pi*x))
    elif motionlaw == 5:
        s, v = L0 + L*(6*x**5 - 15*x**4 + 10*x**3), L/beta*(30*x**4 - 60*x**3 + 30*x**2)
    elif motionlaw == 6:
        s = L0 + L*(-20*x**7 + 70*x**6 - 84*x**5 + 35*x**4)
        v = L/beta*(-140*x**6 + 420*x**5 - 420*x**4 + 140*x**3)
    alpha_max = np.zeros_like(R0_vec, dtype=float)
    for i, R0 in enumerate(R0_vec):
        a = np.arctan2(v, R0 + s)
        alpha_max[i] = np.max(np.abs(a)) * 180/np.pi
    return R0_vec, alpha_max

# Voor elk actief segment plotten — alle drie nu cycloïde
segs_actief = [
    ("Seg 45-80 +10mm  cycloïde", 35,  0, 10, 4),
    ("Seg 90-130 +15mm cycloïde", 40, 10, 25, 4),
    ("Seg 130-180 -25mm cycloïde", 50, 25,  0, 4),
]
plt.figure(figsize=(8,5))
for label, beta, L0, L1, ml in segs_actief:
    R0v, am = Kloomok_Muffley_R0(beta, L0, L1, ml)
    plt.plot(R0v, am, label=label)
plt.axhline(30, ls="--", c="r", label="vuistregel 30°")
plt.xlabel("$R_0$ [mm]"); plt.ylabel(r"$\alpha_\max$ [deg]")
plt.title("Kloomok-Muffley: maximale drukhoek per segment")
plt.grid(); plt.legend(); plt.xlim(0, 150); plt.ylim(0, 80)
plt.show()


Op basis van de grafieken kiezen we $R_0$ zodanig dat alle krommes onder de 30°-lijn blijven.
Bij $R_0 = 70$ mm zit segment 2 (90°-130°, +15 mm) net **op** de grenswaarde —
veiliger is $R_0 = 80$ mm zodat we marge hebben en de drukhoek met excentriciteit verder
omlaag kan.

In [ ]:
R0 = 80.0    # mm — pas indien nodig aan na het bekijken van bovenstaande plot
print(f"Gekozen R0 = {R0} mm")


### 3.2 Volger-radius $R_r$ — controle op ondersnijding

Voor elke actieve beweging zoeken we de minimale kromtestraal $\rho_{\min}$ van de **steekkromme**.
We moeten $R_r < \rho_{\min}$ houden om ondersnijding te vermijden, **én** $R_r$ niet te klein maken
(contactdruk). Standaard: kies $R_r \approx \tfrac{1}{2} R_0$ tot $\tfrac{2}{3} R_0$ als marge het toelaat.

In [ ]:
def Kloomok_Muffley_follower_radius(R0, beta_active_deg, L0, L1, motionlaw):
    beta_vec = np.arange(5, 360, 1.0)
    rho_min = np.zeros_like(beta_vec, dtype=float)
    L = L1 - L0
    for i, b_deg in enumerate(beta_vec):
        beta = b_deg * np.pi/180
        x = np.linspace(0, 1, 200)
        if motionlaw == 1:
            s, v, a = L0*np.ones_like(x), np.zeros_like(x), np.zeros_like(x)
        elif motionlaw == 2:
            s = L0 + L*(3*x**2 - 2*x**3); v = L/beta*(6*x-6*x**2);   a = L/beta**2*(6-12*x)
        elif motionlaw == 3:
            s = L0 + L*(1-np.cos(np.pi*x))/2; v = L/beta*np.sin(np.pi*x)*np.pi/2; a = L/beta**2*np.cos(np.pi*x)*np.pi**2/2
        elif motionlaw == 4:
            s = L0 + L*(x - np.sin(2*np.pi*x)/(2*np.pi)); v = L/beta*(1-np.cos(2*np.pi*x)); a = 2*np.pi*L/beta**2*np.sin(2*np.pi*x)
        elif motionlaw == 5:
            s = L0 + L*(6*x**5 - 15*x**4 + 10*x**3); v = L/beta*(30*x**4-60*x**3+30*x**2); a = L/beta**2*(120*x**3-180*x**2+60*x)
        elif motionlaw == 6:
            s = L0 + L*(-20*x**7+70*x**6-84*x**5+35*x**4)
            v = L/beta*(-140*x**6+420*x**5-420*x**4+140*x**3)
            a = L/beta**2*(-840*x**5+2100*x**4-1680*x**3+420*x**2)
        num = ((R0 + s)**2 + v**2)**1.5
        den = (R0 + s)**2 + 2*v**2 - (R0 + s)*a
        rho = num / np.where(np.abs(den) < 1e-9, 1e-9, den)
        rho_min[i] = np.min(np.abs(rho))
    return beta_vec, rho_min

plt.figure(figsize=(8,5))
for label, beta_a, L0, L1, ml in segs_actief:
    bv, rm = Kloomok_Muffley_follower_radius(R0, beta_a, L0, L1, ml)
    plt.plot(bv, rm, label=label)
    plt.axvline(beta_a, ls=":", alpha=0.5)
plt.xlabel(r"$\beta$ [deg]"); plt.ylabel(r"$\rho_\min$ [mm]")
plt.title(f"Minimale kromtestraal vs segmentbreedte (R0 = {R0} mm)")
plt.grid(); plt.legend(); plt.xlim(0, 360); plt.ylim(0, 200)
plt.show()


Aan de hand van de plot kiezen we $R_r$ kleiner dan de laagste $\rho_{\min}$ over alle
actieve segmenten. Een veilige keuze is $R_r = 15$ mm. De **basiscirkel** is dan $R_b = R_0 - R_r$.

In [ ]:
follower_radius = 15.0          # mm (Rr)
base_radius     = R0 - follower_radius
exc             = 0.0           # mm — kies later iteratief, zie §3.3
print(f"R0 = {R0} mm | Rr = {follower_radius} mm | Rb = {base_radius} mm | exc = {exc} mm")


### 3.3 Drukhoek $\alpha$ en effect van excentriciteit

$$\alpha(\theta) \;=\; \arctan\!\left(\frac{S'(\theta) - e}{\sqrt{R_0^2 - e^2} + S(\theta)}\right)$$

Door $e \ne 0$ te kiezen verschuift de piek-drukhoek tussen rise/fall — we kunnen ze
**symmetrisch** maken om de maximale absolute drukhoek te minimaliseren.

In [ ]:
def calculatePressureAngle(vel, exc, base_radius, follower_radius, lift):
    return np.arctan((vel - exc) /
                     (np.sqrt((base_radius+follower_radius)**2 - exc**2) + lift))

# Vergelijk een paar excentriciteiten
exc_test = [-6, -3, 0, 3, 6]
plt.figure(figsize=(9,4))
for e in exc_test:
    pa = calculatePressureAngle(vel, e, base_radius, follower_radius, lift) * 180/np.pi
    plt.plot(theta_deg, pa, label=f"e = {e} mm  (αmax = {np.max(np.abs(pa)):.1f}°)")
plt.axhline( 30, ls="--", c="r"); plt.axhline(-30, ls="--", c="r")
plt.xlabel("θ [deg]"); plt.ylabel("α [deg]")
plt.title("Drukhoek voor verschillende excentriciteiten")
plt.legend(); plt.grid(); plt.xlim(0, 360)
plt.show()


In [ ]:
# Kies de excentriciteit die αmax minimaliseert (fijne zoektocht)
exc_range = np.arange(-10, 10.01, 0.1)
alpha_max_vec = np.array([
    np.max(np.abs(calculatePressureAngle(vel, e, base_radius, follower_radius, lift)))*180/np.pi
    for e in exc_range
])
exc = float(exc_range[np.argmin(alpha_max_vec)])
print(f"Optimale excentriciteit: e = {exc:.2f} mm  (αmax = {np.min(alpha_max_vec):.2f}°)")

pressure_angle = calculatePressureAngle(vel, exc, base_radius, follower_radius, lift)
plt.figure()
plt.plot(theta_deg, pressure_angle*180/np.pi)
plt.axhline( 30, ls="--", c="r"); plt.axhline(-30, ls="--", c="r")
plt.xlabel("θ [deg]"); plt.ylabel("α [deg]")
plt.title(f"Drukhoek bij e = {exc:.2f} mm")
plt.grid(); plt.xlim(0, 360)
plt.show()


### 3.4 Kromtestraal van de steekkromme en het nokprofiel

We controleren of $\rho_c = \rho - R_r > 0$ overal — anders moeten we $R_r$ verkleinen of $R_0$
vergroten.

In [ ]:
def calculateRadiusCurvature(base_radius, follower_radius, exc, theta, lift, vel, acc):
    d = np.sqrt((base_radius + follower_radius)**2 - exc**2)
    lam   = -theta - np.arctan2(exc, d+lift) + np.pi/2
    dlam  = -1 + exc/((d+lift)**2 + exc**2) * vel
    ddlam = exc/((d+lift)**2 + exc**2)*acc - 2*exc*(d+lift)*vel**2/((d+lift)**2 + exc**2)**2

    g   = np.sqrt((d+lift)**2 + exc**2)
    dg  = vel*(d+lift)/g
    ddg = (vel**2 + acc*(d+lift))/g - (vel**2*(d+lift)**2)/g**3

    h, f = np.sin(lam), np.cos(lam)
    dh, df = np.cos(lam)*dlam, -np.sin(lam)*dlam
    ddh = -np.sin(lam)*dlam**2 + np.cos(lam)*ddlam
    ddf = -np.cos(lam)*dlam**2 - np.sin(lam)*ddlam

    dx, dy = df*g + f*dg, dh*g + h*dg
    ddx, ddy = ddf*g + 2*df*dg + f*ddg, ddh*g + 2*dh*dg + h*ddg

    roc_pitch = -(dx**2 + dy**2)**1.5 / (dx*ddy - dy*ddx)
    roc_cam   = roc_pitch - follower_radius
    return roc_pitch, roc_cam

roc_pitch, roc_cam = calculateRadiusCurvature(base_radius, follower_radius, exc, theta, lift, vel, acc)

# Plot, maar clip om verticale asymptoten te tonen
plt.figure()
plt.plot(theta_deg, np.clip(roc_pitch, -300, 300), label=r"$\rho$ steekkromme")
plt.plot(theta_deg, np.clip(roc_cam,   -300, 300), label=r"$\rho - R_r$ nokprofiel")
plt.axhline(0, c="k", lw=0.5)
plt.xlabel("θ [deg]"); plt.ylabel("Kromtestraal [mm]")
plt.title("Kromtestraal (geclipt op ±300 mm)")
plt.grid(); plt.legend(); plt.xlim(0, 360)
plt.show()

print(f"min ρ_cam (positief deel) = {np.min(roc_cam[roc_cam > 0]):.2f} mm")


### 3.5 Nok-contour

In [ ]:
def plotCamContour(base_radius, follower_radius, exc, theta, lift, pressure_angle, dtheta):
    d = np.sqrt((base_radius + follower_radius)**2 - exc**2)
    lam = -theta - np.arctan2(exc, d+lift) + np.pi/2
    xpitch = np.cos(lam)*np.sqrt((d+lift)**2 + exc**2)
    ypitch = np.sin(lam)*np.sqrt((d+lift)**2 + exc**2)

    xcam = xpitch + follower_radius*np.cos(-theta + pressure_angle + 3*np.pi/2)
    ycam = ypitch + follower_radius*np.sin(-theta + pressure_angle + 3*np.pi/2)

    step = max(1, int(1/dtheta))
    fig, ax = plt.subplots(figsize=(7,7))
    ax.fill(xcam[::step], ycam[::step], color="lightblue", ec="b", label="nok-profiel")
    ax.plot(xpitch[::step], ypitch[::step], "--r", label="steekkromme")
    ax.plot(0, 0, "k+", ms=12, label="rotatiecentrum")
    ax.set_aspect("equal"); ax.grid(); ax.legend(); ax.set_title("Nok-contour [mm]")
    plt.show()

plotCamContour(base_radius, follower_radius, exc, theta, lift, pressure_angle, dtheta)


## 4. Externe statische krachten

Conventie: positieve waarde = drukkracht die de nok ondervindt van de toepassing (vertraagt de
volger op zijn weg omhoog). Negatieve waarde = trekkracht (helpt de volger naar boven).

In [ ]:
ext_load = np.zeros_like(theta)

def addLoadSegment(theta_start, theta_end, F_start, F_end):
    i0, i1 = int(round(theta_start/dtheta)), int(round(theta_end/dtheta))
    seg = theta_deg[i0:i1]
    ext_load[i0:i1] = F_start + (F_end - F_start)/(theta_end - theta_start)*(seg - theta_start)

for s in ext_load_segments:
    addLoadSegment(*s)

plt.figure()
plt.plot(theta_deg, ext_load)
plt.xlabel("θ [deg]"); plt.ylabel("F_ext [N]")
plt.title("Externe statische kracht (positief = drukkracht)")
plt.grid(); plt.xlim(0, 360)
plt.show()


## 5. Krachten op de nok en veer-ontwerp

Vertikaal krachtevenwicht op een **starre** volger (zwaartekracht & wrijving verwaarloosd):
$$
N\,\cos\alpha \;=\; F_{\text{ext}} + (F_{v0} + k_v S) + m\,\omega^2\,\ddot S(\theta)
$$
zodat
$$
N(\theta) = \frac{F_{\text{ext}} + F_{v0} + k_v S + m \omega^2 \ddot S(\theta)}{\cos\alpha}.
$$

Voor **krachtsluiting** moet $N(\theta) \ge 0$ overal. We kiezen eerst $F_{v0}$ (voorspanning) en
bepalen daarna de minimale $k_v$ die overal positieve contactkracht oplevert.

In [ ]:
def calculateForce(rpm, lift, kv, F_v0, pressure_angle, ext_load, mass, acc):
    omg = rpm * 2*np.pi/60
    N_spring = (lift*kv + F_v0) / np.cos(pressure_angle)
    N_load   = ext_load / np.cos(pressure_angle)
    # acc is in mm/rad^2 -> SI: m * (acc/1000) * omg^2  in [N]
    N_acc    = mass * (acc/1000) * (omg**2) / np.cos(pressure_angle)
    N_tot    = N_spring + N_load + N_acc
    Fx = N_tot * np.sin(pressure_angle)
    Fy = N_tot * np.cos(pressure_angle)
    return N_tot, N_acc, N_load, N_spring, Fx, Fy

# --- Eerst kijken hoe N(theta) eruitziet zonder veer ---
N_tot, N_acc, N_load, N_spring, Fx, Fy = calculateForce(
    rpm, lift, 0.0, 0.0, pressure_angle, ext_load, mass_eq, acc)

plt.figure(); plt.plot(theta_deg, N_tot, label="totaal");
plt.plot(theta_deg, N_acc, ":", label="inertie")
plt.plot(theta_deg, N_load, ":", label="ext load")
plt.axhline(0, c="k", lw=0.5)
plt.xlabel("θ [deg]"); plt.ylabel("N [N]"); plt.title("Contactkracht zonder veer")
plt.grid(); plt.legend(); plt.xlim(0, 360); plt.show()


In [ ]:
# Ontwerp van de veer
# Strategie: kies F_v0, bepaal kv zodat N(theta) overal >= N_target

N_target = 50.0   # N — kleine veiligheidsmarge bovenop 0 (vermijdt zero-crossings door numerieke ruis)
F_v0     = 100.0  # N — voorspanning (kies — trade-off: meer voorspanning = hogere contactkrachten overal,
                  #                                   minder = grotere kv nodig)

# Vereiste: (lift*kv + F_v0)*sec(alpha) + N_acc + N_load >= N_target
# => kv >= max over θ van: ((N_target - N_acc - N_load)*cos(alpha) - F_v0) / lift
denom = lift.copy()
denom[denom < 1e-6] = np.nan        # alleen relevant waar S > 0 (alleen daar trekt de veer)
required_kv = ((N_target - N_acc - N_load) * np.cos(pressure_angle) - F_v0) / denom
kv_required = np.nanmax(required_kv)
# Marge
kv = max(0.0, kv_required) * 1.1

print(f"Benodigde kv (theoretisch) = {kv_required:.3f} N/mm")
print(f"Gekozen kv (10% marge)     = {kv:.3f} N/mm")
print(f"Voorspanning F_v0          = {F_v0:.1f} N")

N_tot, N_acc, N_load, N_spring, Fx, Fy = calculateForce(
    rpm, lift, kv, F_v0, pressure_angle, ext_load, mass_eq, acc)

# Check: minimum
i_min = np.argmin(N_tot)
print(f"min N = {N_tot[i_min]:.2f} N bij θ = {theta_deg[i_min]:.1f}°")


In [ ]:
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(theta_deg, N_tot,   label="N totaal")
ax.plot(theta_deg, N_spring,":", label="veer")
ax.plot(theta_deg, N_acc,   ":", label="inertie")
ax.plot(theta_deg, N_load,  ":", label="ext load")
ax.axhline(0, c="k", lw=0.5)
ax.set_xlabel("θ [deg]"); ax.set_ylabel("N [N]")
ax.set_title(f"Contactkracht met veer (kv={kv:.2f} N/mm, F_v0={F_v0:.0f} N)")
ax.grid(); ax.legend(); ax.set_xlim(0, 360)
plt.show()


---

### 5.B Variant: tweestaps aanpak (zoals in Nando's notebook)

> *Gescheiden van de hoofdberekening — ter vergelijking.*

In plaats van eerst $F_{v0}$ te kiezen en daarna $k_v$ in gesloten vorm op te lossen, kan men ook
**twee aparte stappen** uitvoeren:

1. Eis $N(\theta) = 0$ in het globale minimum van de contactkracht (zonder veer): los hieruit $k_v$ op.
2. Plot opnieuw. Als er nog negatieve waarden zijn (typisch in dwell of bij andere extremen), neem
   dan $F_{v0} = -\min N(\theta)$ als voorspanning.

**Wanneer is welke aanpak beter?**

| Criterium | Mijn aanpak (gesloten vorm) | Nando's aanpak (tweestaps) |
|---|---|---|
| Aantal iteraties | 1 berekening | 2 stappen |
| Aantal vrije parameters om te kiezen | $F_{v0}$ vooraf | impliciet — komt uit de berekening |
| Marge op contact ($N \ge N_{\text{target}}$) | direct ingebouwd | moet je achteraf toevoegen |
| Snelheid voor parameter-sweep | direct herberekenen bij andere spec | minder gemakkelijk: 2 stappen |
| Pedagogisch helder | minder — formule "valt uit de lucht" | toont waar elke term vandaan komt |
| Trade-off tussen $k_v$ en $F_{v0}$ expliciet zien | minder | splitsing maakt het zichtbaar |

**Conclusie**: voor het examen is Nando's aanpak makkelijker uit te leggen en toont de fysica
duidelijker. Voor snelle gevoeligheidsanalyses (bv. wat als de massa verandert?) is mijn aanpak
sneller. We tonen hieronder de tweestaps-versie zodat we beide opties paraat hebben.


In [ ]:
# Stap 1: kv zodat N=0 in globaal minimum (zonder veer)
N_tot_noSpring, N_acc_v, N_load_v, _, _, _ = calculateForce(
    rpm, lift, 0.0, 0.0, pressure_angle, ext_load, mass_eq, acc)

i_min = int(np.argmin(N_tot_noSpring))
lift_at_min = lift[i_min]
ext_at_min  = ext_load[i_min]
N_acc_at_min = N_acc_v[i_min]
print(f"Globaal min zonder veer: N = {N_tot_noSpring[i_min]:.2f} N bij theta = {theta_deg[i_min]:.1f} deg  "
      f"(lift = {lift_at_min:.2f} mm)")

if lift_at_min < 1e-3:
    print("(Globaal min ligt in een dwell - kv heeft daar geen invloed; ga direct naar stap 2.)")
    kv_nando = 0.0
else:
    # N = (lift*kv + 0)*sec(alpha) + N_acc + N_load = 0  ->  kv = -(N_acc + N_load)*cos(alpha)/lift
    cos_a = np.cos(pressure_angle[i_min])
    kv_nando = -(N_acc_at_min + N_load_v[i_min]) * cos_a / lift_at_min
    print(f"Stap 1 -> kv (Nando) = {kv_nando:.2f} N/mm  (zonder veiligheidsmarge)")

# Stap 2: voorspanning zodat N >= 0 overal
N_tot_step1, *_ = calculateForce(rpm, lift, kv_nando, 0.0, pressure_angle, ext_load, mass_eq, acc)
F_v0_nando = max(0.0, -np.min(N_tot_step1))
print(f"Stap 2 -> F_v0 (Nando) = {F_v0_nando:.2f} N")

print()
print("Vergelijking:")
print(f"  Mijn aanpak (gesloten vorm, N_target=50 N marge): kv = {kv:.2f} N/mm, F_v0 = {F_v0:.0f} N")
print(f"  Nando-aanpak (tweestaps, N_target=0 N):           kv = {kv_nando:.2f} N/mm, F_v0 = {F_v0_nando:.2f} N")

# Plot beide
N_nando, *_ = calculateForce(rpm, lift, kv_nando, F_v0_nando, pressure_angle, ext_load, mass_eq, acc)
plt.figure(figsize=(10,4))
plt.plot(theta_deg, N_tot,  label=f"Mijn: kv={kv:.1f}, F_v0={F_v0:.0f}  (N_target=50 N)")
plt.plot(theta_deg, N_nando, label=f"Nando: kv={kv_nando:.1f}, F_v0={F_v0_nando:.1f}  (N_target=0)")
plt.axhline(0, c="k", lw=0.5)
plt.xlabel("theta [deg]"); plt.ylabel("N [N]")
plt.title("Veer-ontwerp: mijn aanpak (met marge) vs Nando (zonder marge)")
plt.legend(); plt.grid(); plt.xlim(0, 360)
plt.show()


## 6. Vermogen, koppel — structuur voor vliegwiel en motor

Het mechanische vermogen dat door de nok aan de volger geleverd wordt:
$$
P(\theta) = N(\theta)\,\sin\alpha \cdot R(\theta)\,\omega \quad \text{met } R(\theta) = R_0 + S(\theta).
$$

Het **koppel** dat de motor moet leveren is $T_{\text{motor}}(\theta) = P(\theta)/\omega$.

In [ ]:
# Vermogen en koppel
R_th = R0 + lift                              # mm
P_inst = N_tot * np.sin(pressure_angle) * (R_th/1000) * omega   # W   (R in m)
T_inst = P_inst / omega                                          # Nm

P_avg = np.mean(P_inst)
T_avg = np.mean(T_inst)
P_peak = np.max(np.abs(P_inst))
T_peak = np.max(np.abs(T_inst))

print(f"Gemiddeld vermogen P_avg = {P_avg:.2f} W")
print(f"Piek-vermogen     |P|max = {P_peak:.2f} W")
print(f"Gemiddeld koppel  T_avg  = {T_avg:.2f} Nm")
print(f"Piek-koppel       |T|max = {T_peak:.2f} Nm")

fig, ax = plt.subplots(2, 1, figsize=(9,6), constrained_layout=True)
ax[0].plot(theta_deg, P_inst); ax[0].axhline(P_avg, ls="--", c="r", label=f"gem={P_avg:.1f} W")
ax[0].set_ylabel("P [W]"); ax[0].grid(); ax[0].legend(); ax[0].set_xlim(0, 360)
ax[1].plot(theta_deg, T_inst); ax[1].axhline(T_avg, ls="--", c="r", label=f"gem={T_avg:.2f} Nm")
ax[1].set_ylabel("T [Nm]"); ax[1].set_xlabel("θ [deg]"); ax[1].grid(); ax[1].legend(); ax[1].set_xlim(0, 360)
plt.show()


### 6.1 Vliegwiel-dimensionering

Een vliegwiel buffert de fluctuatie tussen het ogenblikkelijke en het gemiddelde koppel. We
definiëren de **werkfluctuatie** $\Delta W$ als het grootste verschil van $\int (T - T_{\text{avg}})\,d\theta$
over één cyclus. Voor een toelaatbare toerentalfluctuatie $C_s = \Delta\omega/\omega$ geldt:
$$
I_{\text{vliegwiel}} = \frac{\Delta W}{C_s\,\omega^2}.
$$

> **Trade-off (vraag 3)**: kleine $C_s$ → groter vliegwiel → meer massa, meer kosten, langere
> opstarttijd. Voor verpakkingsmachines / kleppensystemen wordt $C_s \approx 0{,}02$–$0{,}05$ vaak
> gehanteerd.
>
> **Let op**: bij **lage rpm** (zoals onze 60 rpm) wordt $I = \Delta W/(C_s \omega^2)$ snel **enorm**
> omdat $\omega$ klein is. Een vliegwiel is dan vaak weinig praktisch — alternatieven zijn:
> *grotere motor*, *meerdere kopieën in parallel met faseverschuiving* (vraag 9!), of
> *het mechanisme via een tandwielreductie sneller laten draaien*.

In [ ]:
Cs = 0.02  # toelaatbare toerentalfluctuatie (2%)

# Werk in functie van theta
dtheta_rad = dtheta * np.pi/180
W_dev = np.cumsum( (T_inst - T_avg) * dtheta_rad )       # integraal van Tdev dθ
DeltaW = np.max(W_dev) - np.min(W_dev)

I_flywheel = DeltaW / (Cs * omega**2)
print(f"ΔW = {DeltaW:.2f} J")
print(f"Vereiste vliegwiel-inertie I = {I_flywheel:.4f} kg·m²  bij Cs = {Cs}")

# Indicatie: massa bij gegeven straal
r_fw = 0.10   # m (10 cm straal vliegwiel)
m_fw = 2 * I_flywheel / r_fw**2          # voor een dunne ring: I = m*r^2; voor schijf: I = m*r^2/2
print(f"Bij vliegwiel-straal {r_fw*1000:.0f} mm: equivalente massa schijf m ≈ {m_fw:.2f} kg")

plt.figure()
plt.plot(theta_deg, W_dev); plt.axhline(np.max(W_dev), ls=":", c="g"); plt.axhline(np.min(W_dev), ls=":", c="r")
plt.xlabel("θ [deg]"); plt.ylabel("∫(T-T_avg) dθ  [J]"); plt.title(f"Werk-deviatie — ΔW = {DeltaW:.1f} J")
plt.grid(); plt.xlim(0, 360); plt.show()


### 6.2 Motor-keuze (structuur)

Een geschikte motor moet voldoen aan:

1. **Piek-koppel** $\ge T_{\max}$ (uit krachtenanalyse).
2. **Nominaal koppel** $\ge T_{\text{rms}}$ (thermische limiet).
3. **Nominaal toerental** $\ge$ 60 rpm (in ons geval).
4. **Mechanisch vermogen** $\ge P_{\text{avg}}$ (met vliegwiel volstaat dit; zonder vliegwiel
   moet de motor $P_{\text{peak}}$ kunnen leveren).

Voor een industriële BLDC- of asynchroon-motor wordt typisch een gear-reducer gebruikt, maar
omdat onze cam-as al traag draait (60 rpm) volstaat directe aandrijving.

**Standaard bijvraag**: schat een motor + jaarlijks energieverbruik.

In [ ]:
T_rms = np.sqrt(np.mean(T_inst**2))
print(f"T_rms = {T_rms:.2f} Nm")
print(f"T_peak = {T_peak:.2f} Nm  →  motor moet minstens dit kunnen leveren (kortstondig)")
print(f"P_avg  = {P_avg:.2f} W  →  vermogen-keuze (met vliegwiel: P_avg, zonder: P_peak)")

# Indicatief: energieverbruik per jaar (continu bedrijf, 8h/dag, 250 dagen)
uren = 8 * 250
E_jaar = P_avg * uren / 1000   # kWh
prijs = 0.25                   # €/kWh (BE 2026 ~)
print(f"Geschat jaarverbruik @ {uren} h/jaar : {E_jaar:.1f} kWh ≈ €{E_jaar*prijs:.0f}/jaar")


## 7. Trillingsanalyse van de nok (flexibele volger)

We modelleren de volger als een 1-DOF massa-veer-demper systeem aangestuurd door de
nok-verplaatsing $u(t)$:

$$
m\,\ddot y + c\,\dot y + k_f\,y = k_f\,u(t)
$$

(De **functionele veer** $k_v$ houdt contact, maar voegt geen extra dynamische stijfheid toe
omdat ze veel slapper is dan de structuur-stijfheid $k_f$ van de volger zelf.)

### 7.1 Structuur-stijfheid $k_f$ bepalen

We kiezen de **kortste rise/fall** als kritisch geval. Voor een goed gedempt response willen we
$\lambda = t_1/t_n \ge \tfrac{0.75}{\zeta}$ (vuistregel) met $t_n = 2\pi/\omega_n$ de natuurlijke
periode. Vaak wordt $\lambda \ge 10$ gekozen voor veiligheid.

In [ ]:
# Vind het kortste niet-dwell segment
actieve = [(s, e, l0, l1, ml) for (s, e, l0, l1, ml) in heffingen if ml != 1]
durations = [(e-s, s, e, l0, l1, ml) for (s, e, l0, l1, ml) in actieve]
b_short, ts, te, l0, l1, ml = min(durations, key=lambda x: x[0])
t1 = T_cycle * b_short/360
print(f"Kortste actief segment: {ts}°–{te}° (Δ{b_short}°), motionlaw {ml}, t1 = {t1*1000:.1f} ms")

lambda_min = 0.75 / zeta_follower
lambda_used = max(10.0, np.ceil(lambda_min))   # kies wat veiliger
omega_n = 2*np.pi*lambda_used / t1
kf = mass_eq * omega_n**2                       # N/m
print(f"λ_min = {lambda_min:.2f}, gekozen λ = {lambda_used}")
print(f"ω_n = {omega_n:.1f} rad/s  (f_n = {omega_n/(2*np.pi):.1f} Hz)")
print(f"Structuur-stijfheid kf = {kf:.0f} N/m = {kf/1000:.2f} N/mm")


### 7.2 Single-rise simulatie

We simuleren de respons van de volger op de kritische rise/fall, en vergelijken met de
nok-input. De **residual vibration** (na $\tau \ge 1$) zit in het verschil.

In [ ]:
# Genormaliseerd: τ = t/t1, θ(τ) = u(t)/h, γ(τ) = y(t)/h
h = abs(l1 - l0)
i0 = int(round(ts/dtheta))
i1 = int(round(te/dtheta))
N_pts = 2000
tau   = np.linspace(0, 2, N_pts)                 # 2 tijdseenheden: 1 voor rise, 1 voor free response
# input theta(tau) — gebruik dezelfde motionlaw als kritisch segment
def motion_theta(tau, motionlaw):
    out = np.zeros_like(tau)
    in_rise = tau <= 1
    x = tau[in_rise]
    if motionlaw == 4:
        out[in_rise] = x - np.sin(2*np.pi*x)/(2*np.pi)
    elif motionlaw == 5:
        out[in_rise] = 6*x**5 - 15*x**4 + 10*x**3
    elif motionlaw == 6:
        out[in_rise] = -20*x**7 + 70*x**6 - 84*x**5 + 35*x**4
    elif motionlaw == 3:
        out[in_rise] = (1 - np.cos(np.pi*x))/2
    else:
        out[in_rise] = 3*x**2 - 2*x**3
    out[tau > 1] = 1
    return out

theta_input = motion_theta(tau, ml)
if l1 < l0:    # fall: spiegel
    theta_input = 1 - theta_input

# Tweede-orde transfer function: gamma/theta = wn^2/(s^2 + 2 ζ wn s + wn^2)
# in dimensieloze tau: tau = t/t1 -> s_tau = s*t1 -> wn_tau = wn*t1 = 2π λ
wn_tau = 2*np.pi*lambda_used
num = [wn_tau**2]
den = [1, 2*zeta_follower*wn_tau, wn_tau**2]
sys = TransferFunction(num, den)

t_out, gamma_out, _ = lsim(sys, U=theta_input, T=tau)

plt.figure(figsize=(9,4))
plt.plot(tau, theta_input, "k--", label=r"$\theta(\tau)$ — nok-input")
plt.plot(t_out, gamma_out, "b",   label=r"$\gamma(\tau)$ — volger-response")
plt.axvline(1, c="gray", ls=":")
plt.xlabel(r"$\tau = t/t_1$"); plt.ylabel("dimensieloze positie")
plt.title(f"Single-rise: motionlaw={ml}, λ={lambda_used}, ζ={zeta_follower}")
plt.grid(); plt.legend()
plt.show()

# Residual vibration amplitude na tau=1
mask_free = t_out > 1.0
A_residual = (np.max(gamma_out[mask_free]) - np.min(gamma_out[mask_free]))/2
print(f"Residual vibration amplitude ≈ {A_residual:.5f} (relatief tov h)")
print(f"In mm:                          ≈ {A_residual*h*1000:.2f} µm")


### 7.3 Multi-rise simulatie (steady-state cyclus)

Hier voeden we de **volledige** bewegingswet aan het 2e-orde systeem en kijken naar de respons na
een aantal cycli (= steady-state).

In [ ]:
# Tijd-as voor één cyclus
t_cycle = theta_deg/360 * T_cycle      # [s]
# Bouw N_cyc cycli aaneen
N_cyc = 8
t_full = np.concatenate([t_cycle + i*T_cycle for i in range(N_cyc)])
u_full = np.tile(lift, N_cyc)          # mm

# Transfer function in echte tijd: wn = sqrt(kf/m)
wn = np.sqrt(kf/mass_eq)
num = [wn**2]
den = [1, 2*zeta_follower*wn, wn**2]
sys_t = TransferFunction(num, den)

t_out, y_out, _ = lsim(sys_t, U=u_full, T=t_full)

# Selecteer de laatste cyclus (steady-state)
mask_last = t_full >= (N_cyc-1)*T_cycle
t_last = t_full[mask_last] - (N_cyc-1)*T_cycle
u_last = u_full[mask_last]
y_last = y_out[mask_last]

fig, ax = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
ax[0].plot(t_full, u_full, "k--", lw=0.5, label="nok-input u(t)")
ax[0].plot(t_full, y_out, "b", lw=0.5, label="volger y(t)")
ax[0].set_xlabel("t [s]"); ax[0].set_ylabel("positie [mm]"); ax[0].legend(); ax[0].grid()
ax[0].set_title("Multi-rise — volledige run (transient + steady-state)")

ax[1].plot(t_last, u_last, "k--", label="nok-input")
ax[1].plot(t_last, y_last, "b", label="volger y(t) — laatste cyclus")
ax[1].plot(t_last, y_last - u_last, "r", lw=0.7, label="fout y-u")
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("positie [mm]"); ax[1].legend(); ax[1].grid()
ax[1].set_title("Steady-state respons (laatste cyclus)")
plt.show()

err = y_last - u_last
print(f"Max steady-state tracking error: {np.max(np.abs(err))*1000:.1f} µm")
print(f"RMS tracking error:              {np.sqrt(np.mean(err**2))*1000:.1f} µm")


### 7.4 Frequentie-inhoud (FFT)

De FFT van de nok-acceleratie laat zien welke harmonischen van $\omega$ de volger excitéren.
Als deze in de buurt van $\omega_n$ liggen → resonantie-risico.

In [ ]:
# FFT van de versnelling (in tijd) - één cyclus volstaat, want periodiek
acc_time = acc_t                       # mm/s² over één cyclus
fs = 1/(dtheta*np.pi/180 / omega)      # samples/s
freqs = np.fft.rfftfreq(len(acc_time), d=1/fs)
mag   = np.abs(np.fft.rfft(acc_time)) / len(acc_time) * 2

plt.figure(figsize=(9,4))
plt.semilogy(freqs, mag)
plt.axvline(omega_n/(2*np.pi), c="r", ls="--", label=f"f_n = {omega_n/(2*np.pi):.1f} Hz")
plt.axvline(1/T_cycle,         c="g", ls=":",  label=f"f_cam = {1/T_cycle:.1f} Hz")
plt.xlabel("f [Hz]"); plt.ylabel("|acc| [mm/s²]")
plt.title("Frequentie-spectrum van de nok-versnelling")
plt.legend(); plt.grid(); plt.xlim(0, 5*omega_n/(2*np.pi))
plt.show()

# Identificeer de dominante harmonischen
i_sort = np.argsort(mag)[::-1][:8]
print("Top harmonischen (f [Hz], magnitude):")
for i in i_sort:
    if freqs[i] > 0:
        print(f"   {freqs[i]:6.2f} Hz  →  {mag[i]:.2e} mm/s²   (harm n = {freqs[i]/(1/T_cycle):.1f})")


### 7.5 Trillings-trade-off: cycloïde vs 7e-graads

We sturen hetzelfde 1-DOF systeem aan met hetzelfde kritische segment, maar in **drie smaken**:
harmonisch, cycloïde (onze keuze), en 7e-graads. We vergelijken:
- piek-versnelling ($\propto$ piek-contactkracht, piek-koppel),
- residual vibration amplitude na de rise.

Dit is **dé** kwantitatieve onderbouwing voor onze keuze (examenvraag 4 én 5).

In [ ]:
# Gebruik dezelfde t1 en lambda als in §7.1; vergelijk drie wetten
def theta_input_law(tau, motionlaw, is_fall=False):
    out = np.zeros_like(tau)
    in_rise = tau <= 1
    x = tau[in_rise]
    if motionlaw == 3:    # harmonisch
        out[in_rise] = (1 - np.cos(np.pi*x))/2
    elif motionlaw == 4:  # cycloïde
        out[in_rise] = x - np.sin(2*np.pi*x)/(2*np.pi)
    elif motionlaw == 6:  # 7e-graads
        out[in_rise] = -20*x**7 + 70*x**6 - 84*x**5 + 35*x**4
    out[tau > 1] = 1
    if is_fall:
        out = 1 - out
    return out

# Piek d²S/dθ² (genormaliseerd) per wet
peak_ddS = {
    "Harmonisch": np.pi**2/2,
    "Cycloïde":   2*np.pi,
    "7e-graads":  7.51,
}

wn_tau = 2*np.pi*lambda_used
num = [wn_tau**2]
den = [1, 2*zeta_follower*wn_tau, wn_tau**2]
sys = TransferFunction(num, den)
tau = np.linspace(0, 2.5, 3000)

fig, ax = plt.subplots(figsize=(10, 5))
results = {}
for name, ml in [("Harmonisch", 3), ("Cycloïde", 4), ("7e-graads", 6)]:
    u = theta_input_law(tau, ml, is_fall=(l1 < l0))
    t_out, y_out, _ = lsim(sys, U=u, T=tau)
    ax.plot(tau, y_out, label=f"{name}  (peak ddS={peak_ddS[name]:.2f})")
    mask = tau > 1.05
    A_res = (np.max(y_out[mask]) - np.min(y_out[mask]))/2
    results[name] = (A_res, peak_ddS[name])
ax.plot(tau, theta_input_law(tau, 4, is_fall=(l1 < l0)), "k--", lw=0.6, label="nok-input (cycloïde)")
ax.axvline(1, c="gray", ls=":")
ax.set_xlabel(r"$\tau = t/t_1$"); ax.set_ylabel(r"$\gamma(\tau)$")
ax.set_title(f"Volger-respons per bewegingswet  (λ={lambda_used}, ζ={zeta_follower})")
ax.legend(); ax.grid()
plt.show()

print("Trade-off tabel:")
print(f"{'Wet':12s}  {'piek d²S/dθ² (norm)':22s}  {'residual amp (rel)':22s}  {'residual (µm)':16s}")
for name, (A, p) in results.items():
    print(f"{name:12s}  {p:22.3f}  {A:22.5f}  {A*h*1000:16.2f}")


**Lezing van de tabel**:

- *Harmonisch* heeft de hoogste residual amplitude — de acceleratiesprong slaat het systeem hard
  aan. Bevestigt waarom we het uitsluiten.
- *Cycloïde* heeft een lage residual; piek-versnelling $2\pi \approx 6{,}28$.
- *7e-graads* heeft de laagste residual (gladde jerk), maar piek-versnelling is $7{,}51$ — ongeveer
  **20% hoger**. Dat vertaalt zich rechtstreeks in **20% hogere piek-contactkracht** op de
  nok-rol, en dus hogere Hertzdruk en zwaardere motor.

**Examen-conclusie**: voor onze toepassing (inertiegedreven, krachten en motorgrootte zijn de
bottleneck, residual van enkele µm is acceptabel) → **cycloïde**. Voor een precisie-toepassing
(positioneringsmachine met tight tolerance) → 7e-graads.

---

## 7.6 Verificatie & uitbreidingen (Nando-stijl)

> *Deze sectie staat los van de hoofdberekening. Drie verfijningen die de
> trillingsanalyse rigoureuzer maken, in de stijl van het voorbeeldnotebook van Nando.*

We doen drie dingen:

1. **7.6.1** — Vergelijk de numerieke free response met de **analytische** gedempte sinus om
   `lsim` te valideren.
2. **7.6.2** — Schat de residual-amplitude **analytisch** via een polynoomspeurder
   $\theta(\tau) = u_\infty + Q (\tau-1)^N / N!$ rond $\tau = 1$, en vergelijk met de numerieke waarde.
3. **7.6.3** — Bereken de **flex-kracht** $F_\text{flex} = k_f (\theta - \gamma) h$ — dit is de
   contactkracht-bijdrage die de structuur-flexibiliteit veroorzaakt, en checkt of $k_v$ ook
   onder die belasting nog volstaat.


### 7.6.1 Analytische free response (verificatie van `lsim`)

Na de rise voert het systeem een vrije gedempte oscillatie uit met natuurlijke frequentie
$\omega_d = \omega_n \sqrt{1 - \zeta^2}$:

$$
\gamma_\text{free}(\tau) = u_\infty + A_1 \, e^{-\zeta \omega_n (\tau-1)} \cos(\omega_d (\tau-1) - \phi)
$$

met $A_1$ en $\phi$ uit de toestand $(\gamma(1), \dot\gamma(1))$.


In [ ]:
# Werk in dimensieloze tau zoals §7.2; gebruik cycloide (onze keuze)
tau_v = np.linspace(0, 3, 5000)

def cycloid_input(tau, is_fall=False):
    out = np.zeros_like(tau)
    in_rise = tau <= 1
    x = tau[in_rise]
    out[in_rise] = x - np.sin(2*np.pi*x)/(2*np.pi)
    out[tau > 1] = 1
    if is_fall:
        out = 1 - out
    return out

u_v   = cycloid_input(tau_v, is_fall=False)
wn_t  = 2*np.pi*lambda_used
num_v = [wn_t**2]
den_v = [1, 2*zeta_follower*wn_t, wn_t**2]
sys_v = TransferFunction(num_v, den_v)
_, gamma_v, _ = lsim(sys_v, U=u_v, T=tau_v)

# Toestand op tau = 1
i1_idx = int(np.argmin(np.abs(tau_v - 1.0)))
x0    = gamma_v[i1_idx]
dgamma = np.gradient(gamma_v, tau_v)
xdot0 = dgamma[i1_idx]

# Vrije oscillatie-formule:  gamma = u_inf + A1 * exp(-z wn (tau-1)) * cos(wd (tau-1) - phi)
u_inf  = float(u_v[-1])
omega_d = wn_t * np.sqrt(1 - zeta_follower**2)
A_cos = x0 - u_inf
A_sin = (xdot0 + zeta_follower*wn_t*A_cos) / omega_d
A1    = np.sqrt(A_cos**2 + A_sin**2)
phi   = np.arctan2(A_sin, A_cos)

tau_free = tau_v[tau_v >= 1]
gamma_analytical = u_inf + A1 * np.exp(-zeta_follower*wn_t*(tau_free-1)) * np.cos(omega_d*(tau_free-1) - phi)
envelope        = A1 * np.exp(-zeta_follower*wn_t*(tau_free-1))

plt.figure(figsize=(10,4))
plt.plot(tau_v,    gamma_v,         "b",  label="numeriek (lsim)")
plt.plot(tau_free, gamma_analytical,"r--",label="analytisch")
plt.plot(tau_free, u_inf+envelope,  "g:", label="omhullende +/-")
plt.plot(tau_free, u_inf-envelope,  "g:")
plt.axvline(1, c="gray", ls=":")
plt.xlabel("tau"); plt.ylabel("gamma(tau)")
plt.title("Free response: numeriek vs analytisch")
plt.legend(); plt.grid()
plt.show()

err_max = np.max(np.abs(gamma_v[tau_v >= 1] - gamma_analytical))
print(f"Max afwijking numeriek vs analytisch op vrije respons: {err_max:.2e}")
print(f"Residual amplitude (analytisch A1) = {A1:.5f} (rel) = {A1*h*1000:.2f} um")


### 7.6.2 Polynoom-approximatie van de residual ($Q$, $N$)

Rond $\tau = 1$ kunnen we het input-signaal benaderen door

$$ \theta(\tau) \approx u_\infty + Q \, \frac{(\tau-1)^N}{N!}. $$

De residual-amplitude is dan ongeveer:

$$ A_{\text{geschat}} \approx \frac{Q}{(2\pi \lambda)^N} \cdot \frac{1}{\sqrt{1-\zeta^2}}. $$

Voor cycloïde geldt $Q = (2\pi)^2$ en $N = 3$ (de derde afgeleide van $\theta(\tau)$ springt
bij $\tau=1$). We checken of die schatting overeenkomt met onze numerieke $A_1$.


In [ ]:
import math

# Cycloide: N=3 (jerk springt), Q = (2 pi)^2
Q_cyc = (2*np.pi)**2
N_cyc = 3
A_est = Q_cyc / ((2*np.pi*lambda_used)**N_cyc) / np.sqrt(1 - zeta_follower**2)
rel_err = (A1 - A_est) / A1 * 100

print(f"Cycloide:  N = {N_cyc},  Q = {Q_cyc:.3f}")
print(f"Numeriek   A1     = {A1:.5f}  (rel)  = {A1*h*1000:.2f} um")
print(f"Analytisch A_est  = {A_est:.5f}  (rel)  = {A_est*h*1000:.2f} um")
print(f"Relatieve fout:    {rel_err:.1f} %")

# Voor 7e-graads: N=5 (5e afgeleide springt)
# De afgeleide formules: theta_7 = -20 x^7 + 70 x^6 - 84 x^5 + 35 x^4
# d^5 theta / dx^5 = -20*7!/(2!)*x^2 + 70*6!/(1!)*x - 84*5!
#                  = -50400 x^2 + 50400 x - 10080
# bij x=1:  -50400 + 50400 - 10080 = -10080
# bij x=0:  -10080
# Constante => N=5 sprong = ... eigenlijk is bij x=1 d5theta/dx5 = -10080
# (Bewuste benadering — exacte Q hangt af van richtingsconventies)
Q_7 = 10080.0
N_7 = 5
A_est_7 = Q_7 / ((2*np.pi*lambda_used)**N_7) / np.sqrt(1 - zeta_follower**2)
print()
print(f"7e-graads (referentie): N = {N_7}, A_est = {A_est_7:.5f} (rel) = {A_est_7*h*1000:.2f} um")
print("=> hogere N => snellere afval met lambda => kleinere residual.")
print()

# Geldigheids-check vuistregel
zeta_lambda = zeta_follower * lambda_used
print(f"Vuistregel-check: zeta*lambda = {zeta_lambda:.3f}  (moet > 0.075 voor < 10% fout)")


### 7.6.3 Krachtanalyse met flexibele volger

Tot nu toe namen we aan dat de volger **star** is. Maar de structuur heeft een eindige stijfheid
$k_f$, dus de volger zelf vervormt. De **flex-kracht** is

$$ F_\text{flex}(t) = k_f \cdot (u(t) - y(t)). $$

Deze kracht komt **bovenop** de inertie- en veer-bijdrage. Een veer die net volstaat in het
starre model kan tekortschieten zodra we de flexibiliteit meenemen.


In [ ]:
# Bereken F_flex over de hele multi-rise steady-state cyclus
# We hebben in §7.3 al y_last (numeriek) en u_last (cam-input) - beide in mm
# kf is in N/m, lift in mm  ->  F_flex = kf * (u-y) * 1e-3
F_flex = (kf/1000.0) * (u_last - y_last)   # N

fig, ax = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
ax[0].plot(t_last, u_last - y_last, "b")
ax[0].set_ylabel("u - y  [mm]"); ax[0].grid()
ax[0].set_title("Volger-vervorming (cam-input minus volger-positie) over een cyclus")

ax[1].plot(t_last, F_flex, "r")
ax[1].axhline(0, c="k", lw=0.5)
ax[1].set_xlabel("t [s]"); ax[1].set_ylabel("F_flex  [N]"); ax[1].grid()
ax[1].set_title(f"Flex-kracht  (kf = {kf/1000:.1f} N/mm)")
plt.show()

print(f"max |F_flex| = {np.max(np.abs(F_flex)):.1f} N")
print(f"RMS  F_flex  = {np.sqrt(np.mean(F_flex**2)):.1f} N")

# Check: blijft N nog positief als we F_flex bovenop de inertie meenemen?
# Project F_flex op de theta_deg-grid via lineaire interpolatie
theta_last_deg = (t_last / T_cycle) * 360
F_flex_on_theta = np.interp(theta_deg, theta_last_deg, F_flex, period=360)
N_with_flex = N_tot - F_flex_on_theta

print(f"min N met F_flex meegenomen: {np.min(N_with_flex):.2f} N")
if np.min(N_with_flex) < 0:
    print(">>> WAARSCHUWING: contact verloren! kv of F_v0 moet hoger.")
else:
    print(">>> OK: contact blijft behouden ook met flex-kracht meegerekend.")


**Interpretatie**: als `min N met F_flex` negatief is, betekent dit dat de eerder gekozen veer
**niet** voldoende is zodra de structuur-flexibiliteit meedoet. In dat geval moet $k_v$ of
$F_{v0}$ omhoog. Voor ons systeem met $k_f \approx 8{,}8$ kN/mm en $\lambda = 10$ is de flex
typisch klein, maar dit is een **kritieke check** voor het examen — Nando heeft hier expliciet
een herziening van zijn veer moeten doen.


## 8. Samenvatting van het ontwerp

| Parameter | Waarde |
|---|---|
| $R_0$ | zie cel — typisch 70 mm |
| $R_r$ | 15 mm |
| $R_b = R_0 - R_r$ | 55 mm |
| Excentriciteit $e$ | bepaald door minimalisatie van $|\alpha|_{\max}$ |
| Max drukhoek $|\alpha|_{\max}$ | $<$ 30° |
| $k_v$ (functionele veer) | uit krachtenanalyse |
| $F_{v0}$ (voorspanning) | 100 N |
| $k_f$ (structuur-stijfheid volger) | uit $\lambda \ge 10$ vuistregel |
| $f_n$ (natuurlijke freq.) | $\omega_n/(2\pi)$ |
| Vliegwiel-inertie | $\Delta W / (C_s \omega^2)$ |
| Motor: $T_{\text{peak}}, T_{\text{rms}}$ | uit koppel-plot |

**Voor het examen** kun je in cel 1 (specificatie) elke parameter aanpassen. Belangrijkste knoppen:

- **Bewegingswet** in `heffingen` — verander motionlaw (1=dwell, 4=cycloid, 5=5e-graads, 6=7e-graads).
- **R0, follower_radius, exc** — geometrie.
- **F_v0, N_target** — veer-trade-off.
- **Cs** — vliegwiel-trade-off.
- **lambda_used** — gevoeligheid voor trillingen.

Veel succes op het examen!
